# Multi-Label Image Classification — Vision Transformer (from Scratch)

Architecture: 16×16 patch embedding → positional encoding → 6-layer Transformer encoder (8 heads, pre-LN) → CLS token → linear head.

ViTs need a lower LR and more epochs to converge from scratch; `LR=3e-4` and cosine annealing are used.

**Edit Section 7 only** when copying this notebook.

**Labels (12 classes):** `pen, paper, book, clock, phone, laptop, chair, desk, bottle, keychain, backpack, calculator`

## 1. Imports & Setup

In [ ]:
import sys
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms

sys.path.append('../')
from eval import LABEL_ORDER, CustomDirectoryLayoutDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')


## 2. Data Loading (Split First, Then Standardization)

Split on raw PIL images first, then apply transforms per subset.

In [ ]:
BASE_DATA_DIR = '../data/aggregated'
IMAGE_SIZE    = 128
BATCH_SIZE    = 128
SPLIT         = [0.8, 0.1, 0.1]   # train / val / test

NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]

NUM_LABELS = len(LABEL_ORDER)
print(f'Labels ({NUM_LABELS}): {LABEL_ORDER}')

full_dataset = CustomDirectoryLayoutDataset(root=BASE_DATA_DIR, transform=None)
n_total = len(full_dataset)
assert n_total > 0, 'Dataset is empty. Check BASE_DATA_DIR path.'

n_train = int(SPLIT[0] * n_total)
n_val   = int(SPLIT[1] * n_total)
n_test  = n_total - n_train - n_val

split_generator = torch.Generator().manual_seed(SEED)
train_raw_subset, val_raw_subset, test_raw_subset = random_split(
    full_dataset, [n_train, n_val, n_test], generator=split_generator
)

print(f'Total samples: {n_total}')
print(f'Train samples: {len(train_raw_subset)}')
print(f'Val samples  : {len(val_raw_subset)}')
print(f'Test samples : {len(test_raw_subset)}')


## 3. Apply Standardization and Build Loaders

In [ ]:
class TransformSubset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, target = self.subset[idx]
        if self.transform is not None:
            image = self.transform(image)
        return image, target

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

train_dataset = TransformSubset(train_raw_subset, transform=train_transform)
val_dataset   = TransformSubset(val_raw_subset,   transform=eval_transform)
test_dataset  = TransformSubset(test_raw_subset,  transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches  : {len(val_loader)}')
print(f'Test batches : {len(test_loader)}')


## 4. Visualize Train Samples (Per Class + Multi-Label)

In [ ]:
PER_CLASS       = 3
MULTI_LABEL_SHOW = 9

def sample_examples_by_class(subset, per_class=3):
    selected = {label: [] for label in LABEL_ORDER}
    for image, target in subset:
        target = target.int()
        for i, label in enumerate(LABEL_ORDER):
            if target[i] == 1 and len(selected[label]) < per_class:
                selected[label].append((image.copy(), target.clone()))
        if all(len(v) >= per_class for v in selected.values()):
            break
    return selected

def get_multi_label_examples(subset, max_items=9):
    items = []
    for image, target in subset:
        if int(target.sum().item()) > 1:
            items.append((image.copy(), target.clone()))
        if len(items) >= max_items:
            break
    return items

def labels_to_text(target):
    labels = [LABEL_ORDER[i] for i, v in enumerate(target) if int(v) == 1]
    return ', '.join(labels) if labels else '(none)'

class_examples = sample_examples_by_class(train_raw_subset, per_class=PER_CLASS)
fig, axes = plt.subplots(len(LABEL_ORDER), PER_CLASS, figsize=(3.2 * PER_CLASS, 2.0 * len(LABEL_ORDER)))
for r, label in enumerate(LABEL_ORDER):
    for c in range(PER_CLASS):
        ax = axes[r, c]
        if c < len(class_examples[label]):
            img, target = class_examples[label][c]
            ax.imshow(img)
            ax.set_title(f'{label}\n{labels_to_text(target)}', fontsize=8)
        else:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center')
        ax.axis('off')
plt.suptitle('Train examples: 2-3 samples per class', fontsize=13)
plt.tight_layout(); plt.show()

multi_examples = get_multi_label_examples(train_raw_subset, max_items=MULTI_LABEL_SHOW)
cols = 3
rows = (len(multi_examples) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))
axes = np.array(axes).reshape(-1)
for idx, ax in enumerate(axes):
    if idx < len(multi_examples):
        img, target = multi_examples[idx]
        ax.imshow(img)
        ax.set_title(labels_to_text(target), fontsize=9)
    ax.axis('off')
plt.suptitle('Train examples with multiple labels', fontsize=13)
plt.tight_layout(); plt.show()


## 5. Metrics Definition

Mirrors the evaluation logic in `eval.py`.

In [ ]:
def compute_multilabel_metrics(labels, preds, probs=None, logits=None):
    labels = labels.float()
    preds  = preds.float()

    if logits is not None:
        loss = nn.BCEWithLogitsLoss()(logits, labels).item()
    elif probs is not None:
        p    = probs.clamp(1e-6, 1 - 1e-6)
        loss = nn.BCELoss()(p, labels).item()
    else:
        loss = float('nan')

    exact_match = (preds == labels).all(dim=1).float().mean().item()
    hamming_acc = (preds == labels).float().mean().item()

    intersection = (preds * labels).sum(dim=1)
    union = ((preds + labels) > 0).float().sum(dim=1)
    iou  = torch.where(union > 0, intersection / union, torch.ones_like(union))
    mean_iou = iou.mean().item()

    tp = ((preds == 1) & (labels == 1)).sum().float()
    fp = ((preds == 1) & (labels == 0)).sum().float()
    fn = ((preds == 0) & (labels == 1)).sum().float()

    precision_micro = (tp / (tp + fp + 1e-8)).item()
    recall_micro    = (tp / (tp + fn + 1e-8)).item()
    f1_micro        = (2 * tp / (2 * tp + fp + fn + 1e-8)).item()

    return {'loss': loss, 'exact_match': exact_match, 'hamming_acc': hamming_acc,
            'mean_iou': mean_iou, 'precision_micro': precision_micro,
            'recall_micro': recall_micro, 'f1_micro': f1_micro}

def evaluate_any_predictor(data_loader, predict_fn, threshold=0.5):
    all_labels, all_preds, all_probs, all_logits = [], [], [], []
    for images, labels in data_loader:
        images = images.to(DEVICE); labels = labels.to(DEVICE)
        preds, probs, logits = predict_fn(images, threshold=threshold)
        all_labels.append(labels.cpu()); all_preds.append(preds.cpu())
        all_probs.append(probs.cpu());   all_logits.append(logits.cpu())
    labels = torch.cat(all_labels, dim=0); preds  = torch.cat(all_preds,  dim=0)
    probs  = torch.cat(all_probs,  dim=0); logits = torch.cat(all_logits, dim=0)
    return compute_multilabel_metrics(labels, preds, probs=probs, logits=logits)

METRIC_KEYS = ['loss', 'exact_match', 'hamming_acc', 'mean_iou',
               'precision_micro', 'recall_micro', 'f1_micro']
print('Metrics:', METRIC_KEYS)


## 6. Baseline Models (Top-K Frequency + Random)

Always predicts a fixed label set regardless of input:
- **Top-1/2/3** — always predict the k most frequent training labels.
- **Random** — each label independently at p=0.5.

Best baseline selected by val F1 and reported on test set.

In [ ]:
train_labels     = torch.cat([targets for _, targets in train_loader], dim=0)
label_prevalence = train_labels.mean(dim=0)

print('Train label prevalence (sorted by frequency):')
sorted_idx = label_prevalence.argsort(descending=True)
for rank, i in enumerate(sorted_idx.tolist()):
    print(f'  #{rank+1:<2} {LABEL_ORDER[i]:<12}  {label_prevalence[i].item():.3f}')

def make_topk_predictor(k):
    topk_idx   = label_prevalence.argsort(descending=True)[:k]
    fixed_pred = torch.zeros(NUM_LABELS, device=DEVICE)
    fixed_pred[topk_idx] = 1.0
    def predict_fn(images, threshold=0.5):
        bsz    = images.shape[0]
        preds  = fixed_pred.unsqueeze(0).expand(bsz, -1).clone()
        probs  = preds * 0.9 + (1 - preds) * 0.1
        logits = torch.logit(probs.clamp(1e-6, 1 - 1e-6))
        return preds, probs, logits
    return predict_fn

def random_baseline_predict(images, threshold=0.5):
    bsz    = images.shape[0]
    probs  = torch.full((bsz, NUM_LABELS), 0.5, device=images.device)
    probs  = probs + 0.01 * torch.randn_like(probs)
    preds  = (probs >= threshold).float()
    logits = torch.logit(probs.clamp(1e-6, 1 - 1e-6))
    return preds, probs, logits

baselines = {
    'top-1 freq': make_topk_predictor(1),
    'top-2 freq': make_topk_predictor(2),
    'top-3 freq': make_topk_predictor(3),
    'random':     random_baseline_predict,
}

def print_metric_table(title, metrics):
    print(f'\n=== {title} ===')
    for k in METRIC_KEYS:
        print(f'  {k:<16} {metrics[k]:.4f}')

best_baseline_name = None; best_baseline_f1 = -1.0
for name, fn in baselines.items():
    m = evaluate_any_predictor(val_loader, fn, threshold=0.5)
    print_metric_table(f'{name}  (Val)', m)
    if m['f1_micro'] > best_baseline_f1:
        best_baseline_f1 = m['f1_micro']; best_baseline_name = name

print(f'\n>>> Best baseline: "{best_baseline_name}"  (F1={best_baseline_f1:.4f})')
best_baseline_test = evaluate_any_predictor(test_loader, baselines[best_baseline_name])
print_metric_table(f'Best baseline "{best_baseline_name}" (Test)', best_baseline_test)


## 7. Model Experiment — Vision Transformer (from Scratch)

`patch_size=16`, `embed_dim=256`, `depth=6`, `num_heads=8`, `mlp_dim=512`.  64 patches per image at 128×128.

In [ ]:
class VisionTransformer(nn.Module):
    """
    Minimal ViT for 128x128 inputs.
    16x16 patches -> 64 tokens + CLS -> 6-layer Transformer encoder
    (8 heads, pre-LayerNorm) -> CLS token -> classifier head.
    """
    def __init__(self, num_classes, image_size=128, patch_size=16,
                 embed_dim=256, num_heads=8, depth=6, mlp_dim=512, dropout=0.1):
        super().__init__()
        assert image_size % patch_size == 0
        self.patch_size  = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        patch_dim        = 3 * patch_size * patch_size

        self.patch_embed = nn.Linear(patch_dim, embed_dim)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed   = nn.Parameter(torch.randn(1, self.num_patches + 1, embed_dim))
        self.dropout     = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=mlp_dim,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=depth)
        self.norm        = nn.LayerNorm(embed_dim)
        self.head        = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B, C, H, W = x.shape; ps = self.patch_size
        patches = x.unfold(2, ps, ps).unfold(3, ps, ps)
        patches = patches.contiguous().view(B, C, -1, ps, ps)
        patches = patches.permute(0, 2, 1, 3, 4).flatten(2)   # (B, N, patch_dim)
        x   = self.patch_embed(patches)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat((cls, x), dim=1)
        x   = self.dropout(x + self.pos_embed)
        x   = self.transformer(x)
        return self.head(self.norm(x[:, 0]))                   # CLS token


def create_model(num_labels: int) -> nn.Module:
    return VisionTransformer(num_classes=num_labels, image_size=IMAGE_SIZE)


model    = create_model(NUM_LABELS).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {n_params:,}')

with torch.no_grad():
    out = model(torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE))
print(f'Output shape: {out.shape}  (expected [2, {NUM_LABELS}])')


In [ ]:
import copy
import itertools

# ── Hyper-parameter search space ─────────────────────────────────────────────
# ViT needs lower LRs and more epochs than a simple CNN
SEARCH_SPACE = {
    'lr':           [1e-3, 3e-4, 1e-4],
    'weight_decay': [0.0,  1e-4, 1e-3],
    'max_epochs':   [25, 35],
}

# ── Fixed training knobs (not searched) ──────────────────────────────────────
WARMUP_EPOCHS       = 5
EARLY_STOP_PATIENCE = 5
LR_REDUCE_PATIENCE  = 3
LR_REDUCE_FACTOR    = 0.5
GRAD_CLIP           = 1.0
THRESHOLD           = 0.5

EXPERIMENT_NAME = 'simple_vit'
CHECKPOINT_DIR  = Path('../checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)
MODEL_PATH = CHECKPOINT_DIR / f'{EXPERIMENT_NAME}.pth'

criterion = nn.BCEWithLogitsLoss()


def train_one_config(config, verbose=False):
    """
    Train a fresh VisionTransformer with the given config dict.
    config keys: lr, weight_decay, max_epochs
    Features: linear LR warmup, ReduceLROnPlateau (post-warmup),
              early stopping with fallback to best weights.
    Returns: (best_state_dict, best_val_f1, history, epochs_run)
    """
    m      = create_model(NUM_LABELS).to(DEVICE)
    lr     = config['lr']
    wd     = config['weight_decay']
    max_ep = config['max_epochs']

    opt = optim.Adam(filter(lambda p: p.requires_grad, m.parameters()),
                     lr=lr, weight_decay=wd)
    scheduler_plateau = optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='max', factor=LR_REDUCE_FACTOR, patience=LR_REDUCE_PATIENCE
    )

    history     = {k: {'train': [], 'val': []} for k in METRIC_KEYS}
    best_val_f1 = -1.0
    best_state  = None
    no_improve  = 0

    for epoch in range(1, max_ep + 1):
        # ── Linear LR warmup ─────────────────────────────────────────────────
        if epoch <= WARMUP_EPOCHS:
            for pg in opt.param_groups:
                pg['lr'] = lr * epoch / WARMUP_EPOCHS

        # ── Train ─────────────────────────────────────────────────────────────
        m.train()
        tr_l, tr_p, tr_pr, tr_lg = [], [], [], []
        for images, targets in train_loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            opt.zero_grad()
            logits = m(images)
            loss   = criterion(logits, targets)
            loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), GRAD_CLIP)
            opt.step()
            with torch.no_grad():
                probs = torch.sigmoid(logits)
                preds = (probs >= THRESHOLD).float()
            tr_l.append(targets.cpu());  tr_p.append(preds.cpu())
            tr_pr.append(probs.cpu());   tr_lg.append(logits.detach().cpu())

        train_metrics = compute_multilabel_metrics(
            torch.cat(tr_l), torch.cat(tr_p),
            probs=torch.cat(tr_pr), logits=torch.cat(tr_lg)
        )

        # ── Validate ─────────────────────────────────────────────────────────
        m.eval()
        val_l, val_p, val_pr, val_lg = [], [], [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(DEVICE); labels = labels.to(DEVICE)
                logits = m(images)
                probs  = torch.sigmoid(logits)
                preds  = (probs >= THRESHOLD).float()
                val_l.append(labels.cpu());  val_p.append(preds.cpu())
                val_pr.append(probs.cpu());  val_lg.append(logits.cpu())

        val_metrics = compute_multilabel_metrics(
            torch.cat(val_l), torch.cat(val_p),
            probs=torch.cat(val_pr), logits=torch.cat(val_lg)
        )

        for k in METRIC_KEYS:
            history[k]['train'].append(train_metrics[k])
            history[k]['val'].append(val_metrics[k])

        val_f1 = val_metrics['f1_micro']

        # ReduceLROnPlateau — only after warmup
        if epoch > WARMUP_EPOCHS:
            scheduler_plateau.step(val_f1)

        # Save best & track patience
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state  = copy.deepcopy(m.state_dict())
            no_improve  = 0
        else:
            no_improve += 1

        if verbose:
            lr_now = opt.param_groups[0]['lr']
            print(f'\nEpoch {epoch:>2}/{max_ep}  [lr={lr_now:.2e}]')
            print(f'  {"Metric":<20} {"Train":>8}  {"Val":>8}')
            print(f'  {"-"*40}')
            for k in METRIC_KEYS:
                print(f'  {k:<20} {train_metrics[k]:>8.4f}  {val_metrics[k]:>8.4f}')

        if no_improve >= EARLY_STOP_PATIENCE:
            if verbose:
                print(f'  [Early stop] No improvement for {EARLY_STOP_PATIENCE} epochs.')
            break

    return best_state, best_val_f1, history, epoch


# ── Grid search ───────────────────────────────────────────────────────────────
keys    = list(SEARCH_SPACE.keys())
configs = [dict(zip(keys, v))
           for v in itertools.product(*[SEARCH_SPACE[k] for k in keys])]

print(f'Grid search: {len(configs)} configurations')
print(f'  lr           : {SEARCH_SPACE["lr"]}')
print(f'  weight_decay : {SEARCH_SPACE["weight_decay"]}')
print(f'  max_epochs   : {SEARCH_SPACE["max_epochs"]}')
print(f'  warmup={WARMUP_EPOCHS}  early_stop_patience={EARLY_STOP_PATIENCE}  '
      f'lr_reduce_patience={LR_REDUCE_PATIENCE}\n')

gs_results = []
for i, cfg in enumerate(configs, 1):
    _, val_f1, _, ep = train_one_config(cfg, verbose=False)
    gs_results.append({'config': cfg, 'val_f1': val_f1, 'epochs': ep})
    print(f'[{i:>2}/{len(configs)}]  lr={cfg["lr"]:.0e}  wd={cfg["weight_decay"]:.0e}'
          f'  max_ep={cfg["max_epochs"]:>2}  →  val_f1={val_f1:.4f}  (ep {ep})')

gs_sorted = sorted(gs_results, key=lambda x: x['val_f1'], reverse=True)
print('\n=== Grid Search Results (ranked by val F1) ===')
print(f'  {"#":<3} {"lr":>8} {"wd":>8} {"max_ep":>7}  {"val_f1":>8}  {"ep":>4}')
print(f'  {"-"*48}')
for rank, r in enumerate(gs_sorted, 1):
    c = r['config']
    print(f'  #{rank:<2}  {c["lr"]:>8.0e}  {c["weight_decay"]:>8.0e}  {c["max_epochs"]:>7}'
          f'  {r["val_f1"]:>8.4f}  {r["epochs"]:>4}')

BEST_CONFIG = gs_sorted[0]['config']
print(f'\nBest config: {BEST_CONFIG}')

# ── Final training with best config ──────────────────────────────────────────
print(f'\n{"="*60}')
print(f'Final training: lr={BEST_CONFIG["lr"]:.0e}  '
      f'wd={BEST_CONFIG["weight_decay"]:.0e}  max_ep={BEST_CONFIG["max_epochs"]}')
print(f'{"="*60}\n')

best_state, best_val_f1, history, epochs_run = train_one_config(BEST_CONFIG, verbose=True)

torch.save(best_state, MODEL_PATH)
model = create_model(NUM_LABELS).to(DEVICE)
model.load_state_dict(best_state)

print(f'\nBest val F1: {best_val_f1:.4f}  checkpoint: {MODEL_PATH}')

# ── Plot all metrics ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
er = range(1, epochs_run + 1)
for i, k in enumerate(METRIC_KEYS):
    axes[i].plot(er, history[k]['train'], label='train')
    axes[i].plot(er, history[k]['val'],   label='val')
    axes[i].set_title(k); axes[i].legend(); axes[i].set_xlabel('Epoch')
axes[-1].axis('off')
plt.suptitle(f'{EXPERIMENT_NAME}  |  best: lr={BEST_CONFIG["lr"]:.0e}  '
             f'wd={BEST_CONFIG["weight_decay"]:.0e}', fontsize=10)
plt.tight_layout(); plt.show()


## 8. Analyze Predictions — Fully Correct / Partially / Incorrect

Load the best checkpoint and split test samples into three buckets. Then show per-class accuracy, confusion matrices and co-occurrence heatmap.

In [ ]:
best_model = create_model(NUM_LABELS).to(DEVICE)
best_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
best_model.eval()
print(f'Loaded: {MODEL_PATH}')

all_images_list, all_labels_list, all_preds_list, all_probs_list = [], [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        logits = best_model(images.to(DEVICE))
        probs  = torch.sigmoid(logits)
        preds  = (probs >= THRESHOLD).float()
        all_images_list.append(images.cpu()); all_labels_list.append(labels.cpu())
        all_preds_list.append(preds.cpu());   all_probs_list.append(probs.cpu())

test_images_all = torch.cat(all_images_list, dim=0)
test_labels_all = torch.cat(all_labels_list, dim=0)
test_preds_all  = torch.cat(all_preds_list,  dim=0)
test_probs_all  = torch.cat(all_probs_list,  dim=0)

match_per_sample = (test_preds_all == test_labels_all)
correct_mask   = match_per_sample.all(dim=1)
incorrect_mask = match_per_sample.sum(dim=1) == 0
partial_mask   = ~correct_mask & ~incorrect_mask

correct_idx   = correct_mask.nonzero(as_tuple=True)[0]
partial_idx   = partial_mask.nonzero(as_tuple=True)[0]
incorrect_idx = incorrect_mask.nonzero(as_tuple=True)[0]

print(f'Fully correct    : {len(correct_idx):>5} / {len(test_labels_all)}')
print(f'Partially correct: {len(partial_idx):>5} / {len(test_labels_all)}')
print(f'Fully incorrect  : {len(incorrect_idx):>5} / {len(test_labels_all)}')

_mean = torch.tensor(NORM_MEAN).view(3, 1, 1)
_std  = torch.tensor(NORM_STD).view(3, 1, 1)

def denorm(t):
    return (t * _std + _mean).clamp(0, 1)

def show_prediction_examples(indices, title, n=4):
    n = min(n, len(indices))
    if n == 0:
        print(f'No examples for "{title}"'); return
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1: axes = [axes]
    for i, idx in enumerate(indices[:n].tolist()):
        img_np = denorm(test_images_all[idx]).permute(1, 2, 0).numpy()
        axes[i].imshow(img_np)
        axes[i].set_title(f'GT:   {labels_to_text(test_labels_all[idx])}\n'
                          f'Pred: {labels_to_text(test_preds_all[idx])}', fontsize=8)
        axes[i].axis('off')
    plt.suptitle(title, fontsize=12, fontweight='bold'); plt.tight_layout(); plt.show()

show_prediction_examples(correct_idx,   'Fully Correct',    n=4)
show_prediction_examples(partial_idx,   'Partially Correct', n=4)
show_prediction_examples(incorrect_idx, 'Fully Incorrect',   n=4)


In [ ]:
L = test_labels_all; P = test_preds_all

per_class_tp = ((P == 1) & (L == 1)).sum(dim=0).float()
per_class_fp = ((P == 1) & (L == 0)).sum(dim=0).float()
per_class_fn = ((P == 0) & (L == 1)).sum(dim=0).float()
per_class_tn = ((P == 0) & (L == 0)).sum(dim=0).float()

per_class_precision = per_class_tp / (per_class_tp + per_class_fp + 1e-8)
per_class_recall    = per_class_tp / (per_class_tp + per_class_fn + 1e-8)
per_class_f1        = 2 * per_class_tp / (2 * per_class_tp + per_class_fp + per_class_fn + 1e-8)
per_class_acc       = (per_class_tp + per_class_tn) / len(L)

print(f'{"Label":<14} {"Acc":>6} {"Prec":>6} {"Rec":>6} {"F1":>6}  {"TP":>5} {"FP":>5} {"FN":>5} {"TN":>5}')
print('-' * 70)
for i, lbl in enumerate(LABEL_ORDER):
    print(f'{lbl:<14} {per_class_acc[i].item():>6.3f} {per_class_precision[i].item():>6.3f} '
          f'{per_class_recall[i].item():>6.3f} {per_class_f1[i].item():>6.3f}  '
          f'{int(per_class_tp[i]):>5} {int(per_class_fp[i]):>5} '
          f'{int(per_class_fn[i]):>5} {int(per_class_tn[i]):>5}')

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(NUM_LABELS), per_class_f1.numpy(), color='steelblue')
ax.set_xticks(range(NUM_LABELS)); ax.set_xticklabels(LABEL_ORDER, rotation=40, ha='right')
ax.set_ylabel('F1'); ax.set_ylim(0, 1); ax.set_title('Per-class F1 on Test Set')
ax.axhline(per_class_f1.mean().item(), color='red', linestyle='--',
           label=f'mean F1 = {per_class_f1.mean().item():.3f}')
ax.legend(); plt.tight_layout(); plt.show()

n_cols = 4; n_rows = (NUM_LABELS + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows))
axes = axes.flatten()
for i, lbl in enumerate(LABEL_ORDER):
    cm = np.array([[int(per_class_tn[i]), int(per_class_fp[i])],
                   [int(per_class_fn[i]), int(per_class_tp[i])]])
    axes[i].imshow(cm, cmap='Blues')
    axes[i].set_title(lbl, fontsize=10, fontweight='bold')
    axes[i].set_xticks([0, 1]); axes[i].set_yticks([0, 1])
    axes[i].set_xticklabels(['Pred 0', 'Pred 1'], fontsize=8)
    axes[i].set_yticklabels(['GT 0', 'GT 1'],     fontsize=8)
    for r in range(2):
        for c in range(2):
            axes[i].text(c, r, str(cm[r, c]), ha='center', va='center', fontsize=11,
                         color='white' if cm[r, c] > cm.max() * 0.5 else 'black')
for j in range(i + 1, len(axes)): axes[j].axis('off')
plt.suptitle('Per-class Binary Confusion Matrices', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

co_matrix = (L.T @ P).numpy()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(co_matrix, cmap='YlOrRd')
plt.colorbar(im, ax=ax, label='count')
ax.set_xticks(range(NUM_LABELS)); ax.set_yticks(range(NUM_LABELS))
ax.set_xticklabels(LABEL_ORDER, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(LABEL_ORDER, fontsize=9)
ax.set_xlabel('Predicted label'); ax.set_ylabel('GT label')
ax.set_title('GT-vs-Prediction co-occurrence\nDiagonal = TP; off-diagonal = confusion', fontsize=11)
for i in range(NUM_LABELS):
    for j in range(NUM_LABELS):
        ax.text(j, i, str(int(co_matrix[i, j])), ha='center', va='center', fontsize=7,
                color='white' if co_matrix[i, j] > co_matrix.max() * 0.6 else 'black')
plt.tight_layout(); plt.show()


## 9. Saliency Maps

Absolute gradient of `logits.sum()` w.r.t. the input — highlights regions the model uses most.

In [ ]:
def compute_saliency(model, image_tensor):
    model.eval()
    inp = image_tensor.unsqueeze(0).to(DEVICE).requires_grad_(True)
    logits = model(inp)
    model.zero_grad()
    logits.sum().backward()
    saliency    = inp.grad.data.abs().squeeze(0)
    saliency, _ = saliency.max(dim=0)
    saliency    = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
    return saliency.cpu()

def show_saliency_examples(indices, title, n=3):
    n = min(n, len(indices))
    if n == 0:
        print(f'No examples for "{title}"'); return
    fig, axes = plt.subplots(n, 2, figsize=(8, 4 * n))
    if n == 1: axes = [axes]
    for i, idx in enumerate(indices[:n].tolist()):
        img_tensor = test_images_all[idx]
        saliency   = compute_saliency(best_model, img_tensor)
        img_np     = denorm(img_tensor).permute(1, 2, 0).numpy()
        axes[i][0].imshow(img_np)
        axes[i][0].set_title(f'GT:   {labels_to_text(test_labels_all[idx])}\n'
                             f'Pred: {labels_to_text(test_preds_all[idx])}', fontsize=8)
        axes[i][0].axis('off')
        axes[i][1].imshow(img_np)
        axes[i][1].imshow(saliency.numpy(), cmap='hot', alpha=0.55)
        axes[i][1].set_title('Saliency overlay', fontsize=8)
        axes[i][1].axis('off')
    plt.suptitle(f'Saliency Maps - {title}', fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()

show_saliency_examples(correct_idx,   'Fully Correct',    n=5)
show_saliency_examples(partial_idx,   'Partially Correct', n=5)
show_saliency_examples(incorrect_idx, 'Fully Incorrect',   n=5)
